# WNBA RTSS Feature Audit

Reproducible review of the local 2023–2025 rich WNBA markings
export. The notebook reads only compact audit artifacts generated
by `wnba_test/audit_rtss_wnba_features.py`; it does not load the
multi-gigabyte source again.

Modeling rule: provider shot-quality outputs remain benchmarks,
and post-shot fields remain targets/diagnostics. They are not
eligible pre-shot predictors.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

INPUT = Path('/Users/russ/Documents/Codex/basketball/nba/research/rapm/wnba_test/Results/rtss_feature_audit')
manifest = json.loads((INPUT / "audit_manifest.json").read_text())
inventory = pd.read_csv(INPUT / "game_inventory.csv")
tables = pd.read_csv(INPUT / "table_coverage.csv")
parity = pd.read_csv(INPUT / "two_d_three_d_parity.csv")
columns = pd.read_csv(INPUT / "column_coverage.csv")
column_catalog = pd.read_csv(INPUT / "column_modeling_catalog.csv")
lineups = pd.read_csv(INPUT / "lineup_quality.csv")
chance_players = pd.read_csv(INPUT / "chance_player_quality.csv")
shots = pd.read_csv(INPUT / "shot_feature_coverage.csv")
identities = pd.read_csv(INPUT / "identity_coverage.csv")
catalog = pd.read_csv(INPUT / "feature_catalog.csv")
manifest

{'source': 'local WNBA RTSS/Second Spectrum 3D markings export',
 'root_label': 'downloads/rtss-data/wnba',
 'games': 761,
 'seasons': {'2023': 173, '2024': 266, '2025': 322},
 'bytes_profiled': 15864510959,
 'game_type_codes': {'101': 21, '102': 682, '104': 57, '105': 1},
 'tables': 32,
 'columns': 934,
 'actor_events': 111956,
 'actor_events_matched_to_pbp': 108551,
 'provider_players': 320,
 'players_mapped_high_confidence': 248,
 'players_mapped_exact': 28,
 'players_ambiguous': 3,
 'players_unmapped': 69,
 'outputs': ['game_inventory.csv',
  'table_coverage.csv',
  'two_d_three_d_parity.csv',
  'column_coverage.csv',
  'column_modeling_catalog.csv',
  'categorical_values.csv',
  'lineup_quality.csv',
  'chance_player_quality.csv',
  'shot_feature_coverage.csv',
  'player_crosswalk.csv',
  'identity_coverage.csv',
  'feature_catalog.csv'],
 'security_note': 'Portal signed URLs were not copied into any output.'}

## Coverage and game inventory

In [2]:
game_types = (
    inventory.groupby(["season", "game_type_code"], as_index=False)
    .agg(games=("game_id", "nunique"))
    .sort_values(["season", "game_type_code"])
)
display(game_types)
display(
    inventory.groupby("season", as_index=False).agg(
        games=("game_id", "nunique"),
        possessions=("possessions", "sum"),
        chances=("chances", "sum"),
        shots=("shots", "sum"),
        passes=("passes", "sum"),
        touches=("touches", "sum"),
    )
)

,season,game_type_code,games
0,2023,101,1
1,2023,102,161
2,2023,104,11
3,2024,101,9
4,2024,102,235
5,2024,104,22
6,2025,101,11
7,2025,102,286
8,2025,104,24
9,2025,105,1


,season,games,possessions,chances,shots,passes,touches
0,2023,173,27505,36661,25460,89497,121820
1,2024,266,42410,56366,39161,135829,184946
2,2025,322,51050,68306,47335,167691,227059


## Lineup integrity and identity crosswalk

In [3]:
display(lineups.round(6))
display(chance_players.round(6))
display(identities.round(6))

,season,possessions,off_exact_5_pct,def_exact_5_pct,both_exact_5_pct,off_unique_pct,def_unique_pct,no_overlap_pct,fully_valid_pct,invalid_possessions
0,2023,27505,0.958517,0.955390,0.931758,1.0,1.0,1.0,0.931758,1877
1,2024,42410,0.964631,0.958783,0.939354,1.0,1.0,1.0,0.939354,2572
2,2025,51050,0.958178,0.954476,0.931557,1.0,1.0,1.0,0.931557,3494


,season,chance_groups,exact_10_rows_pct,exact_10_unique_players_pct,exact_5_offense_pct,exact_5_defense_pct,fully_valid_pct,invalid_chances
0,2023,36661,0.999891,0.999891,0.999945,0.999945,0.999891,4
1,2024,56366,1.000000,1.000000,1.000000,1.000000,1.000000,0
2,2025,68306,1.000000,1.000000,1.000000,1.000000,1.000000,0


,season,provider_players,mapped_high_confidence,mapped_exact,ambiguous,unmapped,mapped_pct
0,2023,162,152,13,2,8,0.938272
1,2024,192,161,7,2,29,0.838542
2,2025,223,184,11,1,38,0.825112


## Major event-table density

In [4]:
key_tables = [
    "possessions", "chances", "shots", "passes", "touches",
    "picks", "off_ball_screens", "rotations", "poss_matchups",
    "shot_contests", "boxout_opportunities", "defender_stance",
]
density = (
    tables[tables["table"].isin(key_tables)]
    .pivot(index="table", columns="season", values="rows_per_game")
    .reindex(key_tables)
)
display(density.round(1))

season,2023,2024,2025
table,,,
possessions,159.0,159.4,158.5
chances,211.9,211.9,212.1
shots,147.2,147.2,147.0
passes,517.3,510.6,520.8
touches,704.2,695.3,705.2
picks,107.3,110.6,115.5
off_ball_screens,120.1,118.8,113.2
rotations,1146.3,1112.4,1129.5
poss_matchups,798.1,799.0,795.6


## 2D base versus 3D enrichment parity

In [5]:
parity_key = parity[parity["table"].isin([
    "possessions", "chances", "shots", "passes", "touches",
    "picks", "rotations", "poss_matchups", "shot_contests",
])]
display(parity_key[[
    "season", "table", "games_2d_nonempty", "games_3d_nonempty",
    "row_count_equal_pct", "common_id_pct_of_2d", "common_id_pct_of_3d",
]].round(6))

,season,table,games_2d_nonempty,games_3d_nonempty,row_count_equal_pct,common_id_pct_of_2d,common_id_pct_of_3d
1,2023,chances,173,169,0.976879,0.976324,1.000000
12,2023,passes,173,169,0.971098,0.976301,0.999989
13,2023,picks,173,169,0.965318,0.974044,0.999945
15,2023,poss_matchups,173,169,0.976879,0.976882,1.000000
16,2023,possessions,173,169,0.976879,0.977022,1.000000
19,2023,rotations,173,169,0.971098,0.971731,1.000000
20,2023,shot_contests,173,169,0.976879,0.977215,1.000000
21,2023,shots,173,169,0.976879,0.976944,1.000000
24,2023,touches,173,169,0.965318,0.976276,1.000000
28,2024,chances,266,266,1.000000,1.000000,1.000000


## Shot-context coverage

In [6]:
pct_columns = [c for c in shots if c.endswith("_coverage")]
display(shots[["season", "shots", *pct_columns]].round(6))

,season,shots,arcAngle_coverage,catchAndShoot_coverage,closestDefDist_coverage,closestDefId_coverage,contestLevel_coverage,contesterIds_coverage,dribblesBefore_coverage,pbpId_coverage,peakShotHeight_coverage,qSP_coverage,qSQ_coverage,region_coverage,releaseTime_coverage,rich3d_match_coverage,rimDepth_coverage,rimLeftRight_coverage,shotClock_coverage
0,2023,25460,0.509387,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.949175,1.0,1.0,1.0,1.0,0.976944,0.509387,0.509387,1.000000
1,2024,39161,0.507801,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.943975,1.0,1.0,1.0,1.0,1.000000,0.507801,0.507801,0.954010
2,2025,47335,0.499546,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.946741,1.0,1.0,1.0,1.0,0.990345,0.499546,0.499546,0.890926


## Schema drift candidates

In [7]:
drift = (
    columns.pivot_table(
        index=["table", "column"],
        columns="season",
        values="coverage",
        aggfunc="max",
    )
    .reset_index()
)
season_cols = [c for c in drift.columns if isinstance(c, int)]
if not season_cols:
    season_cols = [c for c in drift.columns if str(c).isdigit()]
drift["coverage_range"] = drift[season_cols].max(axis=1) - drift[season_cols].min(axis=1)
display(drift.sort_values("coverage_range", ascending=False).head(40).round(6))

season,table,column,2023,2024,2025,coverage_range
237,fake_handoffs,shotClock,1.000000,0.953289,0.877610,0.122390
466,off_ball_screens,shotClock,1.000000,0.949174,0.889793,0.110207
518,passes,shotClock,1.000000,0.954568,0.890908,0.109092
350,handoffs,shotClock,1.000000,0.955020,0.890968,0.109032
900,touches,shotClock,1.000000,0.954327,0.891081,0.108919
584,picks,shotClock,1.000000,0.952737,0.891703,0.108297
433,missed_closeouts,startShotClock,1.000000,0.948405,0.892082,0.107918
419,missed_closeouts,endShotClock,1.000000,0.948405,0.892082,0.107918
216,drives,shotClock,1.000000,0.953603,0.892219,0.107781
385,jumps,endShotClock,1.000000,0.952796,0.892263,0.107737


## Recommended build order

In [8]:
display(catalog.sort_values(["priority", "family"])[
    ["priority", "family", "grain", "model_role", "source_tables", "notes"]
])

,priority,family,grain,model_role,source_tables,notes
9,1,closeout and contest quality,shot/closeout,production candidate,shot_contests; closeouts; missed_closeouts; de...,Best first defensive mechanism family; model M...
8,1,defensive assignments and help,possession/rotation,production candidate,poss_matchups; rotations; chance_players,Keep provider-specific assignments distinct fr...
0,1,lineup exposure,possession/chance-player,production input,possessions; chances; chance_players; player_s...,Canonical 2023-25 lineup and minutes validatio...
1,1,midrange/three shot context,shot,production candidate,shots; shot_contests; touches; chances,Use only MID and 3P per project decision; trai...
2,1,provider shot-quality benchmark,shot,benchmark / semantic audit,shots; touches,Do not ingest as a prior or live feature until...
6,2,passing and advantage creation,pass,production candidate,passes; touches,Supports passer/receiver and creation-pressure...
10,2,rebounding and boxouts,rebound opportunity,production candidate,rebounds; boxout_opportunities; chance_players...,Opportunity-based OREB/DREB and boxout-credit ...
7,2,screening and roll gravity,screen/handoff,production candidate,picks; off_ball_screens; handoffs; fake_handoffs,Mechanism features; estimate player effects wi...
5,2,touch creation and pressure,touch,production candidate,touches; dribbles; drives; isolations; posts,Aggregate prior-game-only to player-season rat...
3,3,shotmaking mechanics,shot,separate research target,shots; jumps,Not pre-shot quality. Keep separate from conte...


## Column-level modeling roles

In [9]:
display(
    column_catalog.groupby(
        ["modeling_role", "recommended_use"], as_index=False
    ).agg(columns=("column", "size"))
    .sort_values("columns", ascending=False)
)
display(
    column_catalog[
        column_catalog["modeling_role"].isin(
            ["provider benchmark", "target / realized outcome", "post-release / pose diagnostic"]
        )
    ][["table", "column", "modeling_role", "recommended_use", "reason"]]
    .sort_values(["modeling_role", "table", "column"])
    .head(80)
)

,modeling_role,recommended_use,columns
1,identifier / join key,join or grouping only,406
2,mechanism candidate,candidate after timing audit,251
0,context candidate,eligible with as-of timing,105
5,review,manual semantic review,80
6,target / realized outcome,target or diagnostic,45
3,post-release / pose diagnostic,separate mechanics research,43
4,provider benchmark,benchmark only,4


,table,column,modeling_role,recommended_use,reason
2,boxout_opportunities,defPlayerFallId,post-release / pose diagnostic,separate mechanics research,Measured during or after the action; not eligi...
10,boxout_opportunities,offPlayerFallId,post-release / pose diagnostic,separate mechanics research,Measured during or after the action; not eligi...
24,chance_players,chancePlayerAroundReboundFallIds,post-release / pose diagnostic,separate mechanics research,Measured during or after the action; not eligi...
25,chance_players,chancePlayerAroundReboundJumpIds,post-release / pose diagnostic,separate mechanics research,Measured during or after the action; not eligi...
26,chance_players,chancePlayerFallIds,post-release / pose diagnostic,separate mechanics research,Measured during or after the action; not eligi...
...,...,...,...,...,...
612,possessions,awayStartScore,target / realized outcome,target or diagnostic,Observed result or label; exclude from pre-eve...
622,possessions,homeStartScore,target / realized outcome,target or diagnostic,Observed result or label; exclude from pre-eve...
628,possessions,outcome,target / realized outcome,target or diagnostic,Observed result or label; exclude from pre-eve...
629,possessions,outcomePbpId,target / realized outcome,target or diagnostic,Observed result or label; exclude from pre-eve...


## Decision

The data is suitable for a new WNBA enrichment layer, but it should
not be poured directly into the current impact metric.

1. Build exact game/event/player crosswalks and a possession/chance
   fact model.
2. Replace 2023–2025 lineup exposure with the rich five-on-five
   source only after possession and stint validation.
3. Train MID/3P context-only expected-points models with
   season-forward, game-grouped evaluation.
4. Add separate defensive contest/rotation, creation/screening, and
   boxout/rebounding summaries as prior-game-only player features.
5. Admit a feature family to VPM/box priors only if it improves an
   untouched future season beyond the existing no-prior and
   accepted-prior lanes.